In [ ]:
import pandas as pd

In [ ]:
df_Trans = pd.read_parquet('transactions.parquet')
df_TransItem = pd.read_parquet('transaction_items.parquet')
df_Users = pd.read_parquet('users.parquet')
df_stores = pd.read_parquet('stores.parquet')
df_menu = pd.read_parquet('menu_items.parquet')
df_voucher = pd.read_parquet('vouchers.parquet')
df_payment = pd.read_parquet('paymentsUnique.parquet')

## Mensamakan tipe data

In [ ]:
df_Trans['transaction_id'] = df_Trans['transaction_id'].astype(str)
df_Trans['user_id'] = (
    pd.to_numeric(df_Trans['user_id'], errors='coerce')
    .astype('Int64')
)
df_Trans['voucher_id']= df_Trans['voucher_id'].astype('Int64')
df_Trans['created_at'] = pd.to_datetime(df_Trans['created_at'], errors='coerce')
df_TransItem['transaction_id'] = df_TransItem['transaction_id'].astype(str)
df_TransItem['created_at'] = pd.to_datetime(df_TransItem['created_at'], errors='coerce')
df_Users['birthdate'] = pd.to_datetime(df_Users['birthdate'], errors='coerce')
df_Users['registered_at'] = pd.to_datetime(df_Users['registered_at'], errors='coerce')
df_Users['user_id'] = (
    pd.to_numeric(df_Users['user_id'], errors='coerce')
    .astype('Int64')
)
df_Users['gender'] = df_Users['gender'].astype(str)
df_menu[['item_name', 'category']] = df_menu[['item_name', 'category']].astype(str)
df_stores[['store_name', 'street', 'city', 'state']] = df_stores[['store_name', 'street', 'city', 'state']].astype(str)
df_stores['postal_code'] = df_stores['postal_code'].astype(object)
df_voucher['valid_from'] = pd.to_datetime(df_voucher['valid_from'])
df_voucher['valid_to'] = pd.to_datetime(df_voucher['valid_to'])


In [ ]:
df_Trans.info()

In [ ]:
df_TransItem.info()

In [ ]:
df_menu.info()

In [ ]:
df_Users.info()

In [ ]:
df_stores.info()

In [ ]:
def run_deep_validation(df_trans, df_items, df_users, df_stores):
    print("--- RUNNING DEEP VALIDATION ---")
    
    # A. Primary Key Uniqueness
    pk_check = df_trans['transaction_id'].duplicated().sum()
    print(f"1. Duplicate Transaction IDs: {pk_check}")
    
    # B. Foreign Key Integrity (Check Store IDs)
    invalid_stores = df_trans[~df_trans['store_id'].isin(df_stores['store_id'])]
    print(f"2. Transaksi dengan Store ID tidak terdaftar: {len(invalid_stores)}")
    
    # C. Impossible Dates Check
    today = pd.Timestamp.now()
    future_trans = df_trans[df_trans['created_at'] > today]
    print(f"3. Transaksi dari 'Masa Depan': {len(future_trans)}")
    
    # D. Age Validity (Mathematical Outlier)
    # Menghitung umur saat registrasi
    temp_users = df_users.dropna(subset=['birthdate', 'registered_at'])
    age_at_reg = (temp_users['registered_at'] - temp_users['birthdate']).dt.days / 365.25
    impossible_age = temp_users[(age_at_reg < 12) | (age_at_reg > 100)]
    print(f"4. User dengan umur tidak masuk akal (<12 atau >100): {len(impossible_age)}")
    
    return pk_check, len(invalid_stores), len(future_trans)

run_deep_validation(df_trans=df_Trans, df_items=df_TransItem, df_users=df_Users, df_stores=df_stores)

In [ ]:
df_Trans.to_parquet( "transactions.parquet")
df_TransItem.to_parquet( "transaction_items.parquet")
df_Users.to_parquet( "users.parquet")
df_stores.to_parquet( "stores.parquet")
df_menu.to_parquet( "menu_items.parquet")
df_voucher.to_parquet("vouchers.parquet")
df_payment.to_parquet("paymentsUnique.parquet")

In [ ]:
import gc
gc.collect()